# Fine-Tuning: Qwen2.5-1.5B-Instruct auf den Encounter-Agent

LoRA/QLoRA-Fine-Tuning von `Qwen/Qwen2.5-1.5B-Instruct`, damit es Groqs `gpt-oss-20b`-Rolle im Encounter-Agent
übernehmen kann (siehe `training/MODEL_CHOICE.md` für die Modellwahl, das LoRA-Schaltplan-Artifact für die
Konzepte dahinter).

**Bevor du loslegst**: `Runtime` -> `Change runtime type` -> `T4 GPU` einstellen, sonst laufen die
4-Bit-Zellen unten nicht.

Konfiguration (begründet in `MODEL_CHOICE.md` bzw. dem gemeinsam besprochenen Plan):
- LoRA-Rang 16, Alpha 32, Dropout 0.05, Ziel-Module `q_proj`/`k_proj`/`v_proj`/`o_proj`
- Lernrate 2e-4, Cosine-Schedule, 3 Epochen mit Early Stopping auf dem Val-Loss
- Train/Val/Test bereits in Schritt 3 gesplittet (217/40/39), auf Setting-Ebene getrennt

## 1. Setup

In [ ]:
!pip install -q -U transformers peft trl bitsandbytes accelerate datasets torchao

In [ ]:
# Sollte eine Tesla T4 mit ~15GB VRAM zeigen. Wenn nicht: Runtime -> Change runtime type -> T4 GPU.
!nvidia-smi

## 2. Trainingsdaten hochladen

Lade die drei Dateien aus `training/artifacts/dataset/` auf deinem Rechner hoch: `train.jsonl`, `val.jsonl`,
`test.jsonl`. Die liegen dort nur lokal (siehe `training/README.md` - absichtlich nicht im Git-Repo).

In [ ]:
from google.colab import files

uploaded = files.upload()  # train.jsonl, val.jsonl, test.jsonl auswählen

In [ ]:
from datasets import load_dataset

dataset = load_dataset(
    "json",
    data_files={"train": "train.jsonl", "validation": "val.jsonl", "test": "test.jsonl"},
)
dataset

## 3. Basismodell laden (4-Bit, QLoRA)

Die eingefrorene Basis wird auf 4-Bit (NF4) komprimiert geladen - das ist das "Q" in QLoRA. Siehe das
LoRA-Schaltplan-Artifact fuer die Begruendung.

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
tokenizer.pad_token = tokenizer.eos_token

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

## 4. LoRA-Adapter aufsetzen

`print_trainable_parameters()` zeigt gleich die echte Zahl - Vergleichswert aus dem Artifact: ~5,5 Mio. von
~1,5 Mrd. (~0,36%).

In [ ]:
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training

model = prepare_model_for_kbit_training(model)

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj"],
    task_type="CAUSAL_LM",
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

## 5. Daten ins Chat-Format bringen

Jede Zeile in `train.jsonl`/`val.jsonl` hat schon die Form `{"messages": [user, assistant], ...}` (siehe
`training/data_gen/dataset.py`). `apply_chat_template` verpackt das in Qwens eigenes Prompt-Format.

In [ ]:
def format_example(example):
    text = tokenizer.apply_chat_template(
        example["messages"], tokenize=False, add_generation_prompt=False
    )
    return {"text": text}


formatted_dataset = dataset.map(format_example)
print(formatted_dataset["train"][0]["text"])

## 6. Training

`EarlyStoppingCallback` bricht ab, sobald der Val-Loss zwei Epochen in Folge nicht mehr sinkt - die Bremse
gegen Auswendiglernen bei nur 217 Trainingsbeispielen (siehe Artifact, Abschnitt "Die Bremse gegen
Auswendiglernen").

Falls `SFTConfig`/`SFTTrainer` sich seit Schreiben dieses Notebooks in `trl` geaendert haben (die API dort
ist nicht immer stabil), zeigt der Fehler meist direkt, welcher Parameter umbenannt wurde - dann kurz
nachschauen, nicht raten.

In [ ]:
from transformers import EarlyStoppingCallback
from trl import SFTConfig, SFTTrainer

training_args = SFTConfig(
    output_dir="qwen2.5-1.5b-enemy-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=4,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=2,  # ~3% von 42 max. Schritten - trl >=1.13 hat kein warmup_ratio mehr
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="eval_loss",
    greater_is_better=False,
    logging_steps=5,
    dataset_text_field="text",
    max_length=512,  # hiess in aelteren trl-Versionen max_seq_length
    report_to="none",
)

trainer = SFTTrainer(
    model=model,
    args=training_args,
    train_dataset=formatted_dataset["train"],
    eval_dataset=formatted_dataset["validation"],
    callbacks=[EarlyStoppingCallback(early_stopping_patience=1)],
)

In [ ]:
trainer.train()

## 7. Sofort speichern

Direkt nach dem Training, bevor irgendwas anderes laeuft: Colab trennt Laufzeiten nach einer Weile Inaktivitaet
von selbst, und dabei geht **alles im Speicher verloren**, inklusive des gerade trainierten Modells - es existiert
bis hierhin nur im GPU-/RAM-Speicher dieser Sitzung, nicht auf Platte. Der Sanity-Check danach ist informativ,
aber optional; das Speichern hier nicht.

Zwei Varianten: der winzige Adapter allein (portabel, wenige MB) und das gemergte Vollmodell (Basis + Adapter
zusammengefuehrt zu normalen Gewichten) - Letzteres brauchen wir in Schritt 7 des Gesamtprojekts fuer die
GGUF-Quantisierung, da llama.cpp/Ollama ein zusammenhaengendes Modell erwartet, nicht Basis+Adapter getrennt.

**Wichtig, per echtem Fehlschlag gelernt**: `merge_and_unload()` direkt auf dem 4-Bit-QLoRA-Trainingsmodell
aufzurufen, dequantisiert nicht zuverlaessig - das Ergebnis kann intern 4-Bit bleiben (erkennbar an
`quantization_config` in der gespeicherten `config.json` und an einer Dateigroesse, die viel zu klein fuer
echtes bf16 ist). Robuster: das Basismodell frisch in voller Praezision laden, den (bereits gespeicherten)
Adapter draufsetzen, dann erst mergen - das umgeht das Dequantisierungsproblem komplett.

In [ ]:
import gc
import os

from peft import PeftModel

model.save_pretrained("qwen2.5-1.5b-enemy-lora-adapter")
tokenizer.save_pretrained("qwen2.5-1.5b-enemy-lora-adapter")

# Reload the base model in full precision (NOT the 4-bit bnb_config from step 3) and
# apply the adapter on top of that clean copy, instead of merging out of the 4-bit
# training model directly - see the note above for why.
del model
gc.collect()
torch.cuda.empty_cache()

base_model_fp16 = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME, dtype=torch.bfloat16, device_map="auto"
)  # older transformers: use torch_dtype= instead of dtype= if this errors
peft_model = PeftModel.from_pretrained(base_model_fp16, "qwen2.5-1.5b-enemy-lora-adapter")
merged_model = peft_model.merge_and_unload()

# Belt and suspenders: make sure no stale 4-bit quantization_config survives into the
# saved config.json even if a future transformers version carries one over.
merged_model.config.quantization_config = None

merged_model.save_pretrained("qwen2.5-1.5b-enemy-merged", safe_serialization=True)
tokenizer.save_pretrained("qwen2.5-1.5b-enemy-merged")

size_gb = sum(
    os.path.getsize(os.path.join("qwen2.5-1.5b-enemy-merged", f))
    for f in os.listdir("qwen2.5-1.5b-enemy-merged")
) / 1e9
print(f"Merged model on disk: {size_gb:.2f} GB (erwartet: ~3 GB fuer bf16, NICHT ~1.6 GB)")

In [ ]:
# Schritt 7 des Gesamtprojekts braucht vor allem den gemergten Ordner (Quantisierung + Ollama).
!zip -r qwen2.5-1.5b-enemy-merged.zip qwen2.5-1.5b-enemy-merged
!zip -r qwen2.5-1.5b-enemy-lora-adapter.zip qwen2.5-1.5b-enemy-lora-adapter

from google.colab import files

files.download("qwen2.5-1.5b-enemy-merged.zip")
files.download("qwen2.5-1.5b-enemy-lora-adapter.zip")

## 8. Sanity-Check: ein paar Beispiele von Hand ansehen

Kein Ersatz fuer den Eval-Harness aus Schritt 2/6 - nur ein schneller "sieht das ueberhaupt vernuenftig aus"-Blick.
Das eigentlich Wichtige (das trainierte Modell) ist an dieser Stelle schon gespeichert - dieser Schritt ist reine
Neugier, kein Risiko mehr, wenn er fehlschlaegt oder die Sitzung danach abbricht.

In [ ]:
merged_model.eval()

for example in dataset["validation"].select(range(3)):
    user_prompt = example["messages"][0]["content"]
    chat = [{"role": "user", "content": user_prompt}]
    input_text = tokenizer.apply_chat_template(chat, tokenize=False, add_generation_prompt=True)
    inputs = tokenizer(input_text, return_tensors="pt").to(merged_model.device)

    output = merged_model.generate(
        **inputs, max_new_tokens=150, temperature=0.9, do_sample=True
    )
    generated = tokenizer.decode(
        output[0][inputs["input_ids"].shape[1] :], skip_special_tokens=True
    )

    print("PROMPT:", user_prompt[:100], "...")
    print("GENERIERT:", generated)
    print("ERWARTET (Groq):", example["messages"][1]["content"])
    print("---")